In [2]:
import torch
import torch.nn as nn
from deviceConfig import device

# To insta relaod modules in Jupyter Notebook
%load_ext autoreload
%autoreload 2

In [3]:
with open("dataset/combined_corpus.txt", "r", encoding="utf-8") as f:
    text = f.read()

In [4]:
len(text)

27226709

In [5]:
# loading the tokenized data

raw_data = torch.load("trained_tokenizer/train.bin")

C:\Users\JAY\AppData\Local\Temp\ipykernel_15048\3281809711.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  raw_data = torch.load("trained_tokenizer/train.bin")


In [6]:
raw_data.shape

torch.Size([27227115])

In [7]:
raw_data[:10]

tensor([ 70, 105, 114, 115, 116,  32,  67, 105, 116, 105])

In [8]:
# We dont need to use a dataloader as the tokenized data is already in a 1D tensor format. We can directly use it for training.

n = int(0.8 * len(raw_data))  # first 80% will be train, rest val

train_data = raw_data[:n]
val_data = raw_data[n:]


In [9]:
from torch.utils.data import Dataset, DataLoader

class GPTDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size

    def __len__(self):
        # The total number of valid starting positions
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        # Slice a context window of length 'block_size'
        x = self.data[idx : idx + self.block_size]
        # Slice the exact same window, shifted one token to the right
        y = self.data[idx + 1 : idx + self.block_size + 1]
        return x, y


train_dataset = GPTDataset(train_data, block_size=256)
val_dataset = GPTDataset(val_data, block_size=256)


In [10]:
from torch.utils.data import DataLoader
import os


Batch_size = 64

train_dataloader = DataLoader(train_dataset, 
                              batch_size=Batch_size, 
                              shuffle=True,
                              num_workers=0
)

val_dataloader = DataLoader(val_dataset,
                            batch_size=Batch_size,
                            shuffle=False,
                            num_workers=0
)

In [11]:
from model_code.moe_gpt import MoEGPT

model_0 = MoEGPT()

state_dict= torch.load("models/MoE_model",weights_only=True)

model_0.load_state_dict(state_dict)

model_0.to(device=device)


MoEGPT(
  (token_embedding_table): Embedding(1500, 384)
  (position_embedding_table): Embedding(256, 384)
  (blocks): ModuleList(
    (0-3): 4 x Block_MoE(
      (sa): MultiHeadAttention(
        (heads): ModuleList(
          (0-3): 4 x Head(
            (key): Linear(in_features=384, out_features=96, bias=False)
            (value): Linear(in_features=384, out_features=96, bias=False)
            (query): Linear(in_features=384, out_features=96, bias=False)
            (dropout): Dropout(p=0.2, inplace=False)
          )
        )
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
      )
      (ffwd): sparseMoEFeedForward(
        (router_1): Router(
          (router): Linear(in_features=384, out_features=4, bias=False)
          (dropout): Dropout(p=0.2, inplace=False)
        )
        (experts): ModuleList(
          (0-3): 4 x expert(
            (net): Sequential(
              (0): Linear(in_features=384, out_

In [12]:
optimizer = torch.optim.AdamW(model_0.parameters(), 
                              lr=3e-4)

In [16]:
next(model_0.parameters()).device

device(type='cuda', index=0)

In [ ]:
from loops_for_model.training_loops import train_loop

train_losses = train_loop(
    model=model_0, 
    data_loader=train_dataloader, 
    optimizer=optimizer, 
    epochs=10, 
    device=device
)
